# 002 - Fixed-Point Substrate (FPS)

**Category:** `physics-inspired`, `gradient-free`  
**Core idea:** Intelligence emerges from fixed points in a learnable medium. Z* = f(Z*). No backprop.  
**Best results:** 96.44% MNIST, 77.15% Fashion-MNIST, 42.24% CIFAR-10

---

## Table of Contents
1. Setup & Imports
2. Algorithm Explanation
3. Training Run (MNIST)
4. Analysis & Visualizations
5. Cross-Dataset Comparison
6. Conclusions & Next Steps

## 1. Setup & Imports

In [ ]:
import sys, os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import json
import time

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from core import FixedPointSubstrate

sys.path.insert(0, os.path.join('..', '..', 'benchmarks'))
from utils import DEVICE

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['font.size'] = 11
print(f'Device: {DEVICE}')

## 2. Algorithm Explanation

### Core Equation

$$Z^* = \tanh\bigl(\sum_s \kappa_s \nabla^2_s Z^* + W_{\text{mix}} Z^* + \alpha X + \beta\bigr)$$

### How It Works

1. **Input projection:** Image passes through 96 learnable 5x5 spatial filters (retinal receptors)
2. **Fixed-point iteration:** The medium iterates Z <- f(Z) with damping + Anderson acceleration until convergence (~7 iterations)
3. **Feature extraction:** Regional mean + variance over a 7x7 grid = 9408 features
4. **Readout:** Two-layer Hebbian classifier (9408 -> 256 -> 10)
5. **Learning:** Error at readout propagated back to medium via local entropy-gated Hebbian updates

### Why Fixed-Point?

- PDE temporal dynamics (v0.1-v0.3) destroyed spatial info via diffusion
- Fixed-point preserves spatial structure: the equilibrium state encodes input features
- Constant memory (no computation graph stored)
- Convergence in 7 iterations (Anderson accelerated)

## 3. Training Run (MNIST)

In [ ]:
from torchvision import datasets, transforms

ds_dir = os.path.join('..', '..', 'datasets')

tf_train = transforms.Compose([
    transforms.RandomAffine(degrees=8, translate=(0.08, 0.08), scale=(0.92, 1.08)),
    transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))
])
tf_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

train_data = datasets.MNIST(ds_dir, train=True, download=True, transform=tf_train)
test_data = datasets.MNIST(ds_dir, train=False, transform=tf_test)
train_loader = torch.utils.data.DataLoader(train_data, 128, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_data, 256, num_workers=2, pin_memory=True)

sub = FixedPointSubstrate(28, 96, 10, 1, 7, 40, 5e-4, 256, str(DEVICE))
print(f'Parameters: {sub.param_count():,}')

In [ ]:
epochs = 30  # Use fewer for notebook demo; full run is 50
lr = 0.012
history = []

for ep in range(1, epochs + 1):
    t0 = time.time()
    asum = nb = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.no_grad():
            a, it = sub.train_step(x, y, lr)
        asum += a; nb += 1
    tra = asum / nb * 100
    
    tsum = tn = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            a, _ = sub.eval_step(x, y)
            tsum += a; tn += 1
    tea = tsum / tn * 100
    dt = time.time() - t0
    history.append({'epoch': ep, 'train': tra, 'test': tea, 'time': dt})
    if ep <= 5 or ep % 5 == 0:
        print(f'Ep {ep:3d} | train {tra:.1f}% | test {tea:.1f}% | {dt:.1f}s')
    lr *= 0.98

print(f'\nBest test: {max(h["test"] for h in history):.2f}%')

## 4. Analysis & Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

eps = [h['epoch'] for h in history]

# Accuracy curve
axes[0].plot(eps, [h['train'] for h in history], label='Train', alpha=0.7)
axes[0].plot(eps, [h['test'] for h in history], label='Test', linewidth=2)
axes[0].axhline(y=98.04, color='red', linestyle='--', alpha=0.5, label='Backprop ref')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Fixed-Point Substrate - MNIST'); axes[0].legend()

# Version evolution
versions = ['v0.1\nPDE', 'v0.2\nPool', 'v0.4a\nFix-Pt', 'v0.4b\n+Var', 'v0.4c\n+Filt', 'v0.5\nFull']
accs = [9.8, 19.8, 76.2, 85.2, 92.7, 96.44]
colors = ['#f85149', '#f85149', '#3fb950', '#3fb950', '#3fb950', '#58a6ff']
axes[1].bar(versions, accs, color=colors)
axes[1].axhline(y=98.04, color='red', linestyle='--', alpha=0.5, label='Backprop')
axes[1].set_ylabel('Test Accuracy (%)'); axes[1].set_title('Version Evolution')
axes[1].legend()

# Timing
axes[2].plot(eps, [h['time'] for h in history], color='orange')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Time (s)')
axes[2].set_title('Time per Epoch')

plt.tight_layout()
plt.savefig('results/training_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Cross-Dataset Results

Full benchmark results (from completed runs):

In [ ]:
# Load saved results
results = {}
for ds in ['mnist', 'fmnist', 'cifar10']:
    path = f'results/{ds}_v05.json'
    if os.path.exists(path):
        with open(path) as f:
            results[ds] = json.load(f)

print('Dataset         | Best     | Baseline | Gap      | Epochs | Params')
print('-' * 72)
for ds, r in results.items():
    gap = r['baseline_value'] - r['best_value']
    print(f"{ds:15s} | {r['best_value']:6.2f}%  | {r['baseline_value']:6.2f}%  | {gap:+6.2f}%  | {r['epochs']:3d}    | {r['total_params']:,}")

## 6. Conclusions & Next Steps

### Results
| Dataset | FPS v0.5 | Backprop | Gap |
|---------|----------|----------|-----|
| MNIST | 96.44% | 98.04% | -1.6% |
| Fashion-MNIST | 77.15% | ~89% | -11.9% |
| CIFAR-10 | 42.24% | 85.83% | -43.6% |

### Key Insights
1. Fixed-point paradigm works: 56% accuracy jump over PDE temporal
2. The medium creates rich features; the bottleneck is reading them
3. CIFAR-10 needs multi-resolution (hierarchical fixed points)
4. Zero backprop, constant memory, 7-iteration convergence

### Next Steps
See README.md - Next Steps / Roadmap